# Function Calls (Tool Calls) 101

Author: Zhaohan Dong

Date: Jan 17, 2025

You might be familiar with basic Grok API operations such as building a chatbot, or letting LLM solve questions based on your conversation input.

With Function Calls (aka Tool Calls), you can expand Grok's capability by interacting with your local system, so Grok can ask your local system perform tasks such as updating a database, call another API to allocate resources, or find information on the website, etc.

### Objectives

By the end of this tutorial, you should be able to set up a basic Function Call for retrieving data and using it with Grok.

## Example Scenario

Imagine it's winter time and you are sitting snugly at your place. You are trying to put together a ski trip with family and friends in 3 days. You are not sure what the weather would be and what's the best way to prepare.

Of course you can Google the weather, but you also want Grok to give you some personalized advices. You set out to build a chatbot with function call that retrieves live weather forecast.

<div style="text-align:center">
<img src="https://upload.wikimedia.org/wikipedia/commons/8/84/Ski_Famille_-_Family_Ski_Holidays.jpg" width=720 style="display: inline-block" />
</div>

## Overview of Function Call (Tool Call)

> Suggested reading: [Function calling](https://docs.x.ai/docs/guides/function-calling) on xAI API Documentation

Letting Grok use function call involves:
1. Defining a function to perform desired actions on your system.
2. Make the function parameter signature available to Grok in your API request, so Grok knows that these functions are available for use on your local system.
3. When Grok determines that your request requires additional information/action available through those functions, it will send a `tool_call` object in the API response message.
4. Your system handles the `tool_call` asked by Grok, and return the result to Grok. (You can also add optional request for Grok)
5. Grok generates response using those results, and might ask for more `tool_calls` based on the specific case.

<div style="text-align:center">
<img src="https://docs.x.ai/assets/function-calling/function-calling-example.png" width=720 style="display: inline-block" />
</div>

In a nutshell — Grok calls our local function in xAI API response, and our local function returns result to Grok via xAI API request.

## Building Blocks

Now with the basic ideas of function calling, let's set up to build the following:
1. Functions that Grok can use
2. Function call handler to handle when Grok asks for a function call in response
3. Set up xAI API chat request and response pipeline

First let's install some dependencies:

In [1]:
!pip install xai-sdk pydantic --quiet

In [2]:
from xai_sdk import Client
from xai_sdk.chat import system, tool, tool_result, user
import os

# Set your API key as an environment variable
XAI_API_KEY = os.getenv("XAI_API_KEY")

# Assumes API key is set via environment variable or other means
client = Client(api_key=XAI_API_KEY)


### 1. Creating Functions locally that Grok can use

To fetch the weather forecast, we will use the [NOAA API Web Service](https://www.weather.gov/documentation/services-web-api).

We can get a 7-day weather forecast on a 2.5km grid area in the following format, by retrieving data from 
`https:///api.weather.gov/gridpoints/{wfo}/{x},{y}/forecast`. The `wfo` and `{x},{y}` can be obtained from `https://api.weather.gov/points/{latitude in signed decimal degrees},{longitude in signed decimal degrees}`
For example, the following is a weather forecast for Boston, MA, retrieved from endpoint `https://api.weather.gov/gridpoints/BOX/72,90/forecast`
```json
{
    "@context": [
        "https://geojson.org/geojson-ld/geojson-context.jsonld",
        {
            "@version": "1.1",
            "wx": "https://api.weather.gov/ontology#",
            "geo": "http://www.opengis.net/ont/geosparql#",
            "unit": "http://codes.wmo.int/common/unit/",
            "@vocab": "https://api.weather.gov/ontology#"
        }
    ],
    "type": "Feature",
    "geometry": {
        "type": "Polygon",
        "coordinates": [
            [
                [
                    -71.029600000000002,
                    42.345599999999997
                ],
                [
                    -71.0244,
                    42.366999999999997
                ],
                [
                    -71.053399999999996,
                    42.370799999999996
                ],
                [
                    -71.058599999999998,
                    42.349399999999996
                ],
                [
                    -71.029600000000002,
                    42.345599999999997
                ]
            ]
        ]
    },
    "properties": {
        "units": "us",
        "forecastGenerator": "BaselineForecastGenerator",
        "generatedAt": "2025-01-17T18:07:01+00:00",
        "updateTime": "2025-01-17T15:18:01+00:00",
        "validTimes": "2025-01-17T09:00:00+00:00/P8DT6H",
        "elevation": {
            "unitCode": "wmoUnit:m",
            "value": 0.91439999999999999
        },
        "periods": [
            {
                "number": 1,
                "name": "This Afternoon",
                "startTime": "2025-01-17T13:00:00-05:00",
                "endTime": "2025-01-17T18:00:00-05:00",
                "isDaytime": true,
                "temperature": 34,
                "temperatureUnit": "F",
                "temperatureTrend": "",
                "probabilityOfPrecipitation": {
                    "unitCode": "wmoUnit:percent",
                    "value": null
                },
                "windSpeed": "5 to 8 mph",
                "windDirection": "NW",
                "icon": "https://api.weather.gov/icons/land/day/few?size=medium",
                "shortForecast": "Sunny",
                "detailedForecast": "Sunny, with a high near 34. Northwest wind 5 to 8 mph."
            },
            {
                "number": 2,
                "name": "Tonight",
                "startTime": "2025-01-17T18:00:00-05:00",
                "endTime": "2025-01-18T06:00:00-05:00",
                "isDaytime": false,
                "temperature": 28,
                "temperatureUnit": "F",
                "temperatureTrend": "",
                "probabilityOfPrecipitation": {
                    "unitCode": "wmoUnit:percent",
                    "value": null
                },
                "windSpeed": "5 to 10 mph",
                "windDirection": "S",
                "icon": "https://api.weather.gov/icons/land/night/sct?size=medium",
                "shortForecast": "Partly Cloudy",
                "detailedForecast": "Partly cloudy, with a low around 28. South wind 5 to 10 mph."
            },
            {
                "number": 3,
                "name": "Saturday",
                "startTime": "2025-01-18T06:00:00-05:00",
                "endTime": "2025-01-18T18:00:00-05:00",
                "isDaytime": true,
                "temperature": 44,
                "temperatureUnit": "F",
                "temperatureTrend": "",
                "probabilityOfPrecipitation": {
                    "unitCode": "wmoUnit:percent",
                    "value": 70
                },
                "windSpeed": "10 to 14 mph",
                "windDirection": "S",
                "icon": "https://api.weather.gov/icons/land/day/rain,30/rain,70?size=medium",
                "shortForecast": "Light Rain Likely",
                "detailedForecast": "Rain likely after 10am. Cloudy, with a high near 44. South wind 10 to 14 mph. Chance of precipitation is 70%. New rainfall amounts less than a tenth of an inch possible."
            },
            {
                "number": 4,
                "name": "Saturday Night",
                "startTime": "2025-01-18T18:00:00-05:00",
                "endTime": "2025-01-19T06:00:00-05:00",
                "isDaytime": false,
                "temperature": 33,
                "temperatureUnit": "F",
                "temperatureTrend": "",
                "probabilityOfPrecipitation": {
                    "unitCode": "wmoUnit:percent",
                    "value": 60
                },
                "windSpeed": "7 to 10 mph",
                "windDirection": "SW",
                "icon": "https://api.weather.gov/icons/land/night/rain,60/bkn?size=medium",
                "shortForecast": "Light Rain Likely then Mostly Cloudy",
                "detailedForecast": "Rain likely and patchy fog before 11pm. Mostly cloudy, with a low around 33. Southwest wind 7 to 10 mph. Chance of precipitation is 60%. New rainfall amounts less than a tenth of an inch possible."
            },
            // ...
        ]
    }
}
```

For our use case, we will limit the forecast area to a few popular ski areas, and return the `properties.periods` from the API response from NOAA to Grok.

Here, we will define the function inputs and outputs using Pydantic:

In [3]:
from enum import Enum
from typing import Literal
from pydantic import BaseModel, Field
import requests

# Available ski resorts
class SkiResort(str, Enum):
    aspen = 'aspen'
    breckenridge = 'breckenridge'
    jackson_hole = 'jackson_hole'
    vali = 'vali'

# Tool call request available to Grok
class ForecastRequest(BaseModel):
    location: SkiResort = Field(description="Ski resort location name in snake case")


# Probability of precipitation used in response body definition
class ProbabilityOfPrecipitation(BaseModel):
    unitCode: str = Field(description="Unit code of precipitation")
    value: int | None = Field(description="Probability of precipitation in unitCode")

# Response format to send back to Grok
class ForecastResponse(BaseModel):
    number: int = Field(description="Index of the forecast in the sequence")
    name: str = Field(description="Name of the report period, relative to today")
    startTime: str = Field(description="ISO8601 format of forecasting period start with timezone")
    endTime: str = Field(description="ISO8601 format of forecasting period end with timezone")
    isDaytime: bool = Field(description="Whether forecasting period is daytime. True if it is daytime")
    temperature: int = Field(description="Temperature in temperatureUnit unit")
    temperatureUnit: Literal["C", "F"] = Field(description="Temperature Unit")
    temperatureTrend: str = Field(description="Description of temperature trend")
    probabilityOfPrecipitation: ProbabilityOfPrecipitation = Field(description="Probability of Precipitation")
    windSpeed: str = Field(description="Description of Wind Speed")
    windDirection: str = Field(description="Wind direction")
    shortForecast: str = Field(description="A short summary of forecast condition")
    detailedForecast: str = Field(description="Detailed description of the forecast")


# URLs of the forecast locations
skiResortForecastUrl: dict[SkiResort, str] = {
    SkiResort.aspen : "https://api.weather.gov/gridpoints/GJT/156,102/forecast",
    SkiResort.breckenridge : "https://api.weather.gov/gridpoints/BOU/25,53/forecast",
    SkiResort.jackson_hole : "https://api.weather.gov/gridpoints/RIW/42,139/forecast",
    SkiResort.vali : "https://api.weather.gov/gridpoints/GJT/173,121/forecast"
}

# Local function that will be executed when Grok asks for
def get_weather_forecast(**kwargs) -> list[ForecastResponse]:
    req = ForecastRequest(**kwargs)  # Validate and parse the keyword parameters that Grok sends to us
    forecast_url = skiResortForecastUrl[req.location]  # Get request url for a given ski resort location

    forecast = requests.get(url=forecast_url).json()  # Retrieve forecast

    res: list[ForecastResponse] = []

    for item in forecast["properties"]["periods"]:
        item.pop("icon")  # Remove unnecessary weather icon url
        res.append(item)
    return res

You can preview the data we send to Grok, when Grok asks for the weather forecast at Aspen, CO:

In [4]:
get_weather_forecast(location='aspen')

[{'number': 1,
  'name': 'Tonight',
  'startTime': '2025-09-05T18:00:00-06:00',
  'endTime': '2025-09-06T06:00:00-06:00',
  'isDaytime': False,
  'temperature': 47,
  'temperatureUnit': 'F',
  'temperatureTrend': '',
  'probabilityOfPrecipitation': {'unitCode': 'wmoUnit:percent', 'value': 24},
  'windSpeed': '5 mph',
  'windDirection': 'SE',
  'shortForecast': 'Slight Chance Showers And Thunderstorms then Partly Cloudy',
  'detailedForecast': 'A slight chance of showers and thunderstorms before 9pm. Partly cloudy, with a low around 47. Southeast wind around 5 mph. Chance of precipitation is 20%.'},
 {'number': 2,
  'name': 'Saturday',
  'startTime': '2025-09-06T06:00:00-06:00',
  'endTime': '2025-09-06T18:00:00-06:00',
  'isDaytime': True,
  'temperature': 74,
  'temperatureUnit': 'F',
  'temperatureTrend': '',
  'probabilityOfPrecipitation': {'unitCode': 'wmoUnit:percent', 'value': 50},
  'windSpeed': '0 to 5 mph',
  'windDirection': 'W',
  'shortForecast': 'Chance Showers And Thunder

The function to call is defined! Hurray! Now we need to send the function name and parameters signature so that Grok knows how to call the function.

In [5]:
# Definition of parameters with Pydantic JSON schema
tools_definition = [
    tool(
        name="get_weather_forecast",  # the function name that we defined
        description="Get the weather forecast at a given location",  # Description of the function, so that Grok knows whether the function would be useful to solving the problem
        parameters=ForecastRequest.model_json_schema() # Generate the request parameter schema from Pydantic
    ),
]

### 2. Function handler to invoke the function we defined and add result to conversation history

With our previous definition of the function, we can send a request to Grok.

Let's see how Grok will respond:

In [6]:
chat = client.chat.create(
    model="grok-3",
    messages=[],
    tools=tools_definition,
)
chat.append(user("What should I prepare for a ski trip on Wednesday according to the weather in Vali, CO?"))
response = chat.sample()

# You can inspect the response which contains a tool call
response

id: "9b6edf0e-0eb7-4638-98f3-47e0c1335407_us-east-1"
choices {
  finish_reason: REASON_TOOL_CALLS
  message {
    role: ROLE_ASSISTANT
    tool_calls {
      id: "call_03405308"
      function {
        name: "get_weather_forecast"
        arguments: "{\"location\":\"vali\"}"
      }
    }
  }
}
created {
  seconds: 1757121136
  nanos: 280062220
}
model: "grok-3"
system_fingerprint: "fp_898ae9f31c"
usage {
  completion_tokens: 27
  prompt_tokens: 326
  total_tokens: 353
  prompt_text_tokens: 326
  cached_prompt_text_tokens: 3
}

You can see in the response, Grok included the tool_calls.

We need to design a handler to handle the `tool_calls`, by:
1. Add Grok's response to chat history, in case we want to continue the conversation.
2. Decide if Grok's response has a `tool_call`. If not:
    - Print Grok's response message to end user.
    - Skip the following steps.
3. Calling function named `get_weather_forecast`.
4. Add the function result to chat history.

In [7]:
import json
from typing import Callable

# Define a mapping between function name and the function's callable object
tools_map: dict[str, Callable] = {
    "get_weather_forecast": get_weather_forecast
}

def function_calls_handler(chat, response):
    # Add Grok's response to chat
    chat.append(response)

    # Check if there is any tool calls in response
    if response.tool_calls:
        # There is a tool call, run get_weather_forecast or iterate through all of them
        for tool_call in response.tool_calls:

            # Get the tool function name and arguments Grok wants to call
            function_name = tool_call.function.name
            function_args = json.loads(tool_call.function.arguments)

            # Call one of the tool function defined earlier in tools_map with arguments
            result = tools_map[function_name](**function_args)

            # Append the result from tool function call to the chat
            chat.append(tool_result(json.dumps(result)))

    else:
        # No tool call, print the message content
        print(response.content)

Now we call the handler on our earlier message from Grok and see what happens to our chat

In [8]:
# Call handler
function_calls_handler(chat, response)

**Now we can finally send this back to Grok to get the recommendation!**

### 3. Setting up the whole pipeline to generate recommendation

As a recap:
- We have defined our local function `get_weather_forecast()` that retrieves NOAA weather forecast for Grok to run.
- We have defined a `function_calls_handler()` to append Grok's response to chat, and run the function if Grok asks for it.
  - As part of this, we also defined a `tools_map` to map function name given by Grok -> function Callable object.

We will run our chat request/response up to this point:

In [9]:
# 1. Send initial request to Grok, Grok responds with a tool_call request
chat = client.chat.create(
    model="grok-3",
    messages=[],
    tools=tools_definition,
)
chat.append(user("What should I prepare for a ski trip on Wednesday according to the weather in Vali, CO?"))
response = chat.sample()

# 2. We handle the tool_call, and append Grok's response + the function result to chat
function_calls_handler(chat, response)

# 3. We send back to Grok to get our final recommendation
response = chat.sample()

# 4. Print the response
function_calls_handler(chat, response)

For your ski trip on Wednesday in Vail, CO, here's what you should prepare for based on the weather forecast:

- **Temperature**: Expect a high of 71°F during the day, dropping to a low of 42°F at night. These temperatures are quite mild for skiing, so you might not need your heaviest winter gear.
- **Conditions**: The day will be partly sunny with a slight chance of showers and thunderstorms after noon (24% probability of precipitation). This means there could be some wet or variable conditions on the slopes.
- **Wind**: Winds will be around 5 mph from the south-southwest, which shouldn't pose much of a challenge.
- **Nighttime**: There's a slight chance of showers and thunderstorms continuing into the evening (17% probability), with mostly cloudy skies.

**Preparation Tips**:
- **Clothing**: Layer up with moisture-wicking base layers, a waterproof or water-resistant ski jacket, and pants to handle potential showers. Bring a mid-layer for warmth during cooler morning or evening hours,

You can wrap a user input, the call to Grok, and function_call_handler in a loop. This way, Grok will continuously answer your questions!